In [1]:
import os
import requests
from datetime import datetime, date
import time
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv

# ---------------------------
# ENV
# ---------------------------
load_dotenv()

POSTGRES_DB = os.getenv("POSTGRES_DB")
POSTGRES_USER = os.getenv("POSTGRES_USER")
POSTGRES_PASSWORD = os.getenv("POSTGRES_PASSWORD")
POSTGRES_HOST = os.getenv("POSTGRES_HOST", "localhost")
POSTGRES_PORT = os.getenv("POSTGRES_PORT", "5432")
POLYGON_API_KEY = os.getenv("POLYGON_API_KEY")

if not all([POSTGRES_DB, POSTGRES_USER, POSTGRES_PASSWORD, POLYGON_API_KEY]):
    raise EnvironmentError("Faltan variables de entorno")

# ---------------------------
# Conexión PostgreSQL
# ---------------------------
engine = create_engine(
    f"postgresql://{POSTGRES_USER}:{POSTGRES_PASSWORD}@{POSTGRES_HOST}:{POSTGRES_PORT}/{POSTGRES_DB}"
)

# ---------------------------
# Configuración filtros opciones
# ---------------------------
MIN_DTE = 25
MAX_DTE = 45
MIN_DELTA = -0.35
MAX_DELTA = -0.25
CONTRACT_TYPE = "put"

# ---------------------------
# 1️⃣ Obtener tickers multifactor top 20%
# ---------------------------
query_tickers = """
SELECT DISTINCT ticker
FROM volatilidad.multifactor_top20
"""
df_tickers = pd.read_sql(query_tickers, engine)
tickers_validos = df_tickers["ticker"].tolist()

print(f"Procesando {len(tickers_validos)} tickers multifactor top20...")

# ---------------------------
# 2️⃣ Consulta Polygon
# ---------------------------
def buscar_opciones_por_ticker(ticker_raiz):
    hoy = datetime.now()
    url = f"https://api.polygon.io/v3/snapshot/options/{ticker_raiz}?limit=250&apiKey={POLYGON_API_KEY}"

    resultados = []
    buscando = True

    while url and buscando:
        try:
            r = requests.get(url)
            if r.status_code != 200:
                print(f"Error Polygon {ticker_raiz}: {r.status_code}")
                break

            data = r.json()
            contracts = data.get("results", [])

            for c in contracts:
                details = c.get("details", {})
                greeks = c.get("greeks", {})
                day = c.get("day", {})

                if details.get("contract_type") != CONTRACT_TYPE:
                    continue

                expiry = details.get("expiration_date")
                if not expiry:
                    continue

                fecha_vto = datetime.strptime(expiry, "%Y-%m-%d")
                dte = (fecha_vto - hoy).days

                if dte > MAX_DTE:
                    buscando = False
                    break

                delta = greeks.get("delta")
                if delta is None or not (MIN_DELTA <= delta <= MAX_DELTA):
                    continue

                if MIN_DTE <= dte <= MAX_DTE:
                    resultados.append({
                        "ticker": ticker_raiz,
                        "opcion": details.get("ticker"),
                        "contract_type": details.get("contract_type"),
                        "strike": details.get("strike_price"),
                        "vto": expiry,
                        "dte": dte,

                        # Greeks
                        "delta": round(delta, 4),
                        "gamma": greeks.get("gamma"),
                        "theta": greeks.get("theta"),
                        "vega": greeks.get("vega"),

                        # Volatilidad y liquidez
                        "iv": c.get("implied_volatility"),
                        "oi": c.get("open_interest", 0),
                        "volume": day.get("volume", 0),
                        "vwap": day.get("vwap"),
                        "close_price": day.get("close"),

                        "fecha": date.today()
                    })

            next_url = data.get("next_url")
            url = f"{next_url}&apiKey={POLYGON_API_KEY}" if next_url else None

        except Exception as e:
            print(f"Excepción {ticker_raiz}: {e}")
            break

    return resultados

# ---------------------------
# 3️⃣ Procesar e insertar
# ---------------------------
total_insertados = 0

for ticker in tickers_validos:
    print(f"\nProcesando {ticker}...")
    data = buscar_opciones_por_ticker(ticker)

    if data:
        df = pd.DataFrame(data)
        df.to_sql(
            "multifactor_top20_opciones",
            engine,
            schema="volatilidad",
            if_exists="append",
            index=False
        )
        print(f"{len(df)} opciones insertadas")
        total_insertados += len(df)
    else:
        print("No se encontraron PUTs válidos")

    time.sleep(0.25)

print(f"\nProceso finalizado. Total opciones insertadas: {total_insertados}")


Procesando 296 tickers multifactor top20...

Procesando META...
8 opciones insertadas

Procesando FG...
No se encontraron PUTs válidos

Procesando KKRS...
No se encontraron PUTs válidos

Procesando RGA...
1 opciones insertadas

Procesando AMRX...
1 opciones insertadas

Procesando SNEX...
2 opciones insertadas

Procesando DECK...
6 opciones insertadas

Procesando BLD...
1 opciones insertadas

Procesando BCC...
1 opciones insertadas

Procesando MGEE...
2 opciones insertadas

Procesando BBWI...
3 opciones insertadas

Procesando DBX...
2 opciones insertadas

Procesando ESTC...
No se encontraron PUTs válidos

Procesando EE...
1 opciones insertadas

Procesando CNXC...
1 opciones insertadas

Procesando GWW...
3 opciones insertadas

Procesando HQY...
1 opciones insertadas

Procesando IONS...
1 opciones insertadas

Procesando V...
4 opciones insertadas

Procesando ITW...
No se encontraron PUTs válidos

Procesando TDW...
No se encontraron PUTs válidos

Procesando DIOD...
No se encontraron PUTs v

1 opciones insertadas

Procesando AEP...
No se encontraron PUTs válidos

Procesando MAS...
No se encontraron PUTs válidos

Procesando SNV...
2 opciones insertadas

Procesando PSA...
No se encontraron PUTs válidos

Procesando HHH...
1 opciones insertadas

Procesando MKSI...
2 opciones insertadas

Procesando DPZ...
1 opciones insertadas

Procesando WES...
1 opciones insertadas

Procesando CMCSA...
2 opciones insertadas

Procesando CHTR...
4 opciones insertadas

Procesando TVTX...
1 opciones insertadas

Procesando RS...
1 opciones insertadas

Procesando BLKB...
No se encontraron PUTs válidos

Procesando COP...
5 opciones insertadas

Procesando NOC...
3 opciones insertadas

Procesando XOM...
4 opciones insertadas

Procesando KMX...
No se encontraron PUTs válidos

Procesando NXST...
No se encontraron PUTs válidos

Procesando VCYT...
1 opciones insertadas

Procesando ETSY...
3 opciones insertadas

Procesando BLDR...
1 opciones insertadas

Procesando FRT...
1 opciones insertadas

Procesando A